# 가상 참여자 노트북 (중복을 그대로 두고 셔플 KFold로 여러 모델 비교)

In [1]:
import pandas as pd
import numpy as np
import joblib
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape, test.shape)

(1279, 13) (320, 12)


In [2]:
def display_scores(scores):
    print('Scores:', scores)
    print('Mean:', scores.mean())
    print('Standard deviation:', scores.std())

In [3]:
X, y = train.drop(columns=['id', 'quality']), train['quality']
X_test = test.drop(columns=['id'])
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
cv = KFold(n_splits=10, shuffle=True, random_state=42)
candidates = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=0),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=300, random_state=0),
    'GradientBoosting': GradientBoostingRegressor(random_state=0),
}
results = {}
for name, m in candidates.items():
    s = np.sqrt(-cross_val_score(m, X, y, scoring='neg_mean_squared_error', cv=cv))
    results[name] = s
    print(name, 'CV RMSE 평균', round(s.mean(), 4))
best = min(results, key=lambda k: results[k].mean())
print('가장 좋은 모델:', best)
display_scores(results[best])
model = candidates[best].fit(X, y)
pred = model.predict(X_test)

Ridge CV RMSE 평균 0.6552


RandomForest CV RMSE 평균 0.5819


ExtraTrees CV RMSE 평균 0.5687


GradientBoosting CV RMSE 평균 0.6255
가장 좋은 모델: ExtraTrees
Scores: [0.52882248 0.53383733 0.56895029 0.70221939 0.54202873 0.62251401
 0.54318737 0.55961094 0.58954277 0.49624002]
Mean: 0.5686953315475384
Standard deviation: 0.05537038196625271


In [4]:
pd.DataFrame({'id': test['id'], 'quality': pred}).to_csv('submission.csv', index=False)
joblib.dump(model, 'model.pkl')

['model.pkl']